In [ ]:
# Mesh retrieved from: https://geometric-kernels.github.io/GeometricKernels/examples/frontends/GPyTorch.html

In [ ]:
# Import the Mesh space and the general-purpose MaternGeometricKernel
from geometric_kernels.spaces.mesh import Mesh

# Stuff
import numpy as np

import plotly.graph_objects as go

from pathlib import Path

import networkx as nx

INFO (geometric_kernels): Numpy backend is enabled. To enable other backends, don't forget to `import geometric_kernels.*backend name*`.
INFO (geometric_kernels): We may be suppressing some logging of external libraries. To override the logging policy, call `logging.basicConfig`.
INFO (geometric_kernels): Torch backend enabled.


# Mesh Plotting Utils for plotly

In [2]:
def update_figure(fig):
    """Utility to clean up figure"""
    fig.update_layout(scene_aspectmode="cube")
    fig.update_scenes(xaxis_visible=False, yaxis_visible=False, zaxis_visible=False)
    # fig.update_traces(showscale=False, hoverinfo="none")
    fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))

    fig.update_layout(plot_bgcolor="rgba(0,0,0,0)", paper_bgcolor="rgba(0,0,0,0)")
    fig.update_layout(
        scene=dict(
            xaxis=dict(showbackground=False, showticklabels=False, visible=False),
            yaxis=dict(showbackground=False, showticklabels=False, visible=False),
            zaxis=dict(showbackground=False, showticklabels=False, visible=False),
        )
    )
    return fig

def plot_mesh(mesh: Mesh, vertices_colors = None, **kwargs):
    plot = go.Mesh3d(
        x=mesh.vertices[:, 0],
        y=mesh.vertices[:, 1],
        z=mesh.vertices[:, 2],
        i=mesh.faces[:, 0],
        j=mesh.faces[:, 1],
        k=mesh.faces[:, 2],
        intensity=vertices_colors,
        **kwargs
    )
    return plot

# Extracting Mesh

In [3]:
mesh = Mesh.load_mesh(str(Path.cwd().parent / "Geometric Kernels_TeddyBear" / "teddy.obj"))
print("Number of vertices in the mesh:", mesh.num_vertices)

Number of vertices in the mesh: 1598


In [ ]:
# Save mesh geometry
np.save('Data/teddy_vertices.npy', mesh.vertices)
np.save('Data/  teddy_faces.npy', mesh.faces)

print("Also saved:")
print(f"teddy_vertices.npy ({mesh.vertices.shape})")
print(f"teddy_faces.npy ({mesh.faces.shape})")

✅ Also saved:
   📄 teddy_vertices.npy ((1598, 3))
   📄 teddy_faces.npy ((3192, 3))


In [5]:
# Define the camera
camera = dict(
    up=dict(x=0, y=1, z=0),
    center=dict(x=0, y=0, z=0),
    eye=dict(x=0, y=0.7, z=1.25)
)

plot = plot_mesh(mesh)
fig = go.Figure(plot)
update_figure(fig)
fig.update_layout(
    scene_camera=camera
)
fig.show()


# Compute the Geodesic Distances

In [ ]:

print("Computing Geodesic Distance Matrix for Teddy Mesh")
print("=" * 70)

# ============================================================================
# 1. LOAD MESH
# ============================================================================
mesh = Mesh.load_mesh(str(Path.cwd().parent / "Geometric Kernels_TeddyBear" / "teddy.obj"))
vertices   = mesh.vertices
faces      = mesh.faces
n_vertices = mesh.num_vertices

print(f"Mesh loaded: {n_vertices:,} vertices, {len(faces):,} faces")

# ============================================================================
# 2. BUILD MESH GRAPH
# ============================================================================
def build_mesh_graph(vertices, faces):
    G = nx.Graph()
    G.add_nodes_from(range(len(vertices)))
    for face in faces:
        face = list(face)
        n = len(face)
        for k in range(n):
            u = face[k]
            v = face[(k + 1) % n]
            if u != v and not G.has_edge(u, v):
                weight = float(np.linalg.norm(vertices[u] - vertices[v]))
                G.add_edge(u, v, weight=weight)
    return G

print("\nConstructing mesh graph...")
mesh_graph = build_mesh_graph(vertices, faces)
print(f"Graph: {mesh_graph.number_of_nodes():,} nodes, "
      f"{mesh_graph.number_of_edges():,} edges")
print(f"Connected: {nx.is_connected(mesh_graph)}")

# ============================================================================
# 3. COMPUTE FULL GEODESIC DISTANCE MATRIX
# ============================================================================
print("\n Computing geodesic distances...")

geo_dist = np.full((n_vertices, n_vertices), np.inf, dtype=np.float32)
np.fill_diagonal(geo_dist, 0.0)

for source in range(n_vertices):
    dists = nx.shortest_path_length(mesh_graph, source=source, weight='weight')
    for target, d in dists.items():
        if target >= source:                    # upper triangle
            geo_dist[source, target] = d
            geo_dist[target, source] = d        # mirror immediately

    if (source + 1) % 100 == 0:
        print(f"  Progress: {source+1}/{n_vertices} ({100*(source+1)/n_vertices:.1f}%)")

# ============================================================================
# 4. VERIFY AND SAVE
# ============================================================================
print(f"\n Geodesic distance matrix computed!")
print(f"   Shape:     {geo_dist.shape}")
print(f"   Min (off-diagonal): {geo_dist[geo_dist > 0].min():.6f}")
print(f"   Max:       {geo_dist.max():.4f}")
print(f"   Mean:      {geo_dist.mean():.4f}")
print(f"   Symmetric: {np.allclose(geo_dist, geo_dist.T)}")
print(f"   Any inf:   {np.any(np.isinf(geo_dist))}")

np.save('Data/teddy_exact_geodesic_distances_graph.npy', geo_dist)
print(f"\n Saved: teddy_exact_geodesic_distances_graph.npy")
print("=" * 70)